In [0]:
%run ./env

In [0]:
%run ./python_libraries

# phase2_migration_schema — filet de sécurité, à exécuter une fois par environnement

La phase 2 ajoute des colonnes à deux tables existantes du modèle :

| Table | Colonnes ajoutées |
|---|---|
| `dim_batches_specifications` | `id_good_specy`, `id_good_variety`, `id_parameter_production_type`, `id_requirement_specification` |
| `fact_batch_note` | `key_location`, `key_impact`, `key_event`, `key_detail` |

## Est-ce vraiment nécessaire ?

**Cela dépend d'une configuration Spark, pas du code de `delta_function`.**

Le code de `handle_table_update` ne demande explicitement aucune évolution de
schéma : ni `option("mergeSchema", "true")` sur le `write.mode("append")` du mode
`full`, ni `withSchemaEvolution()` sur le MERGE du mode `update`. Mais Databricks
propose un réglage global qui l'active pour les deux :

```
spark.databricks.delta.schema.autoMerge.enabled
```

- **S'il est à `true`** sur le cluster ou le workspace, l'ajout de colonnes se
  fait tout seul, dans les deux modes. Ce notebook ne fera alors rien : les
  colonnes existeront déjà, ou seront créées au premier run.
- **S'il est à `false`** (la valeur par défaut de Spark), le premier run échoue
  sur une erreur de schéma, et ce notebook est indispensable.

La cellule suivante affiche la valeur effective. Si tu as déjà ajouté des
colonnes à ces tables par un simple run en mode `full`, c'est que le réglage est
actif dans ton environnement.

Ce notebook reste utile comme filet de sécurité : il est **idempotent** (il ne
touche qu'aux colonnes réellement absentes) et il garantit que le pipeline
démarre aussi sur un environnement où le réglage ne serait pas actif — dev ou
preprd, par exemple.

`ALTER TABLE ... ADD COLUMNS` est une opération de métadonnées sur Delta : elle
ne réécrit pas les données et les lignes existantes prennent `NULL` sur les
nouvelles colonnes. Ces `NULL` seront remplis au premier passage du pipeline.

In [0]:
# Réglage qui décide si l'évolution de schéma est automatique.
# Sur Serverless certaines configurations sont verrouillées : le get peut échouer.
try:
    auto_merge = spark.conf.get("spark.databricks.delta.schema.autoMerge.enabled", "false")
except Exception as e:
    auto_merge = f"non lisible ({e})"

print(f"spark.databricks.delta.schema.autoMerge.enabled = {auto_merge}")
print()
if str(auto_merge).lower() == "true":
    print("-> L'évolution de schéma est automatique : ce notebook ne fera probablement rien.")
else:
    print("-> L'évolution de schéma n'est PAS automatique : ce notebook est nécessaire")
    print("   avant le premier run du pipeline modifié.")

In [0]:
target_dim_batches_specifications = f"{current_catalog}.{current_schema}.dim_batches_specifications"
target_fact_batch_note = f"{current_catalog}.{current_schema}.fact_batch_note"

colonnes_a_ajouter = {
    target_dim_batches_specifications: [
        ("id_good_specy", "INT"),
        ("id_good_variety", "INT"),
        ("id_parameter_production_type", "INT"),
        ("id_requirement_specification", "INT"),
    ],
    target_fact_batch_note: [
        ("key_location", "STRING"),
        ("key_impact", "STRING"),
        ("key_event", "STRING"),
        ("key_detail", "STRING"),
    ],
}

In [0]:
for table_name, colonnes in colonnes_a_ajouter.items():
    if not spark.catalog.tableExists(table_name):
        print(f"{table_name} : table absente, rien à faire (elle sera créée au premier run).")
        continue

    existantes = set(spark.table(table_name).columns)
    manquantes = [(nom, typ) for nom, typ in colonnes if nom not in existantes]

    if not manquantes:
        print(f"{table_name} : déjà à jour.")
        continue

    ddl = ", ".join(f"{nom} {typ}" for nom, typ in manquantes)
    spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({ddl})")
    print(f"{table_name} : colonnes ajoutées -> {ddl}")

## Contrôle

Les colonnes doivent apparaître, à `NULL` sur les lignes existantes tant que le
pipeline n'a pas été relancé.

In [0]:
for table_name in colonnes_a_ajouter:
    if spark.catalog.tableExists(table_name):
        print(f"--- {table_name}")
        display(spark.table(table_name).limit(5))